In [ ]:
import os
from scraper import fetch_website_contents
import requests
from openai import OpenAI
from IPython.display import Markdown, display

requests.get("http://localhost:11434").content

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [3]:
job_post = fetch_website_contents("https://justjoin.it/job-offer/shimi-sp-z-o-o--test-specialist-sharepoint-m-f-n--warszawa-net")

system_prompt = """
You are a highly efficient Job Finder Assistant. Your primary responsibility is to analyze job posting web links (or provided text) and help the user extract key skills, requirements, and specific details from the posting.

### Critical Response Rule (Targeted Queries vs. Full Summaries)

1. **Targeted Questions (STRICT):** 
   - If the user asks a specific question about the job (e.g., "What is the offered employment type?", "What English level is required?", "Is this remote?", "What is the salary?"):
   - **ANSWER ONLY THAT QUESTION.** 
   - Provide a direct, concise 1-2 sentence response containing only the requested information. 
   - DO NOT output the full template, background info, or extra skill lists unless explicitly asked.

2. **General Link/Posting Submissions:**
   - If the user simply shares a link or pastes a job description without asking a specific targeted question, analyze the post and provide the full structured summary below.

---

### Output Format (Used ONLY for General Link Submissions):

## [Job Title] at [Company Name]

### 📌 Role Summary
- Seniority/Experience Level: [e.g., 3-5 years / Senior]
- Location/Work Type: [e.g., Remote / On-site / Hybrid]
- Employment Type: [e.g., Full-time / Part-time / Contract]
- Language Requirements: [e.g., Fluent English (C1/C2), Native German]

### 🛠️ Key Hard & Technical Skills
- [Skill 1]
- [Skill 2]

### 🤝 Key Soft Skills & Competencies
- [Skill 1]
- [Skill 2]

### 💡 Core Requirements & Nice-to-Haves
- Must-Haves: [Top 2-3 non-negotiable requirements]
- Nice-to-Haves/Preferred: [Bonus qualifications or optional tools]

---

### Interaction Rules & Safeguards:
- Missing Information: If the requested specific detail (e.g., salary, English level) is NOT mentioned in the job post, explicitly state: "This detail is not specified in the job posting."
- Invalid Links: If a link is broken or paywalled, politely ask the user to paste the raw text of the job description instead.
- Accuracy: Never invent or assume requirements not grounded in the text.
"""

user_prompt = """ Analyze this job position """


In [6]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ]
messages_for(job_post)

def job_details(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "gpt-oss",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
job_details("https://justjoin.it/job-offer/shimi-sp-z-o-o--test-specialist-sharepoint-m-f-n--warszawa-net")

In [ ]:
def display_job_details(url):
    key_details = job_details(url)
    display(Markdown(key_details))

display_job_details("https://justjoin.it/job-offer/shimi-sp-z-o-o--test-specialist-sharepoint-m-f-n--warszawa-net")